# 10 — So sánh với dự báo có sẵn (CAMS / Open-Meteo)

Câu hỏi: GRU và Transformer chính xác hơn dự báo PM2.5 đang được công bố **bao nhiêu %**?

| Mốc so sánh | Nguồn | Ghi chú |
|---|---|---|
| **CAMS forecast** | ECMWF/Copernicus, kho dự báo ADS (`src/fetch_benchmarks.py cams`) | Dự báo đúng như lúc phát hành, có horizon. Phát hành 07:00 và 19:00 giờ VN. Đây là mô hình đứng sau dự báo PM2.5 của Open-Meteo, Windy và nhiều ứng dụng |
| **CAMS hiệu chỉnh** | CAMS + hồi quy tuyến tính theo từng horizon, học trên Validation | Mốc mạnh hơn: CAMS đã được chỉnh lệch cho riêng Hoàn Kiếm |
| **Open-Meteo (CAMS)** | API Open-Meteo, không cần key (`src/fetch_benchmarks.py openmeteo`) | Chuỗi ghép từ các lần chạy mới nhất (lead ngắn nhất) nên là mốc **có lợi cho CAMS** |
| Persistence | PM2.5 hiện tại | Mốc tối thiểu |

Quy tắc công bằng: mô hình của ta chỉ dùng dữ liệu có sẵn lúc dự báo (bản "vận hành" từ 07/08), cùng giai đoạn Test, cùng đáp án (PM2.5 Weatherbit tại Hoàn Kiếm), cùng thời điểm phát hành,
cùng horizon. CAMS được coi là có sẵn ngay lúc khởi tạo (thực tế trễ vài giờ), nên so sánh này nghiêng về phía CAMS.

In [ ]:
# Chạy trên Colab: bỏ comment, sửa link repo
# !git clone <link-repo> repo
# %cd repo/notebooks

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import forecasting as fc

BENCH_DIR = ROOT / 'data' / 'external' / 'benchmarks'
REPORT_DIR = ROOT / 'results' / 'report'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODELS = {'GRU': 'gru', 'Transformer': 'transformer'}
SPLITS = fc.load_splits(ROOT)

## I. Gom mọi dự báo về dạng (issue_time, horizon)

In [ ]:
truth = pd.read_csv(ROOT / 'data' / 'processed' / 'feature_data.csv', parse_dates=['Local Time'],
                    usecols=['Local Time', 'PM25']).set_index('Local Time')['PM25']

def load_model(d, split):
    p = pd.read_csv(ROOT / 'results' / 'models' / d / f'predictions_{split}.csv',
                    parse_dates=['issue_time', 'target_time'])
    return p.set_index(['issue_time', 'horizon'])

# Dùng bản "vận hành" trên Test nếu có: mô hình chỉ nhận dữ liệu có sẵn lúc dự báo thật, giống CAMS
frames = {}
for name, d in MODELS.items():
    op = ROOT / 'results' / 'models' / d / 'predictions_test_operational.csv'
    test_split = 'test_operational' if op.exists() else 'test'
    print(name, '-> Test dùng', f'predictions_{test_split}.csv')
    frames[name] = pd.concat([load_model(d, 'validation'), load_model(d, test_split)])
base = frames['GRU'][['target_time', 'y_true']].copy()
forecasts = pd.DataFrame({name: f['y_pred'] for name, f in frames.items()})

# Persistence: PM2.5 tại thời điểm phát hành
forecasts['Persistence'] = truth.reindex(base.index.get_level_values('issue_time')).to_numpy()

# Open-Meteo (CAMS, lead ngắn nhất): giá trị tại thời điểm mục tiêu
om = pd.read_csv(BENCH_DIR / 'openmeteo_cams_pm25.csv', parse_dates=['time_local']).set_index('time_local')
forecasts['Open-Meteo (CAMS)'] = om['openmeteo_cams_pm25'].reindex(base['target_time']).to_numpy()

# CAMS forecast từ ADS (nếu đã tải)
cams_path = BENCH_DIR / 'cams_forecast_pm25.csv'
HAS_CAMS = cams_path.exists()
if HAS_CAMS:
    cams = pd.read_csv(cams_path, parse_dates=['issue_time']).set_index(['issue_time', 'horizon'])
    forecasts['CAMS forecast'] = cams['cams_pm25'].reindex(base.index).to_numpy()
else:
    print('Chưa có', cams_path.name, '-> chạy: py -3.11 src/fetch_benchmarks.py cams  (cần tài khoản ADS)')

table = base.join(forecasts)
print(table.shape)
display(table.head())

## II. Mốc "CAMS hiệu chỉnh"

CAMS có độ phân giải ~40 km nên thường lệch so với một điểm trong nội đô. Học hệ số `PM2.5 ≈ a + b·CAMS`
riêng cho từng horizon trên **Validation**, rồi áp lên Test. Mô hình của ta cũng chỉ dùng Validation để chọn cấu hình,
nên hai bên có lượng thông tin như nhau.

In [ ]:
raw_bench = 'CAMS forecast' if HAS_CAMS else 'Open-Meteo (CAMS)'
corrected = f'{raw_bench} hiệu chỉnh'
issue = table.index.get_level_values('issue_time')
is_val = (issue >= SPLITS['Validation'][0]) & (issue <= SPLITS['Validation'][1])
is_test = (issue >= SPLITS['Test'][0]) & (issue <= SPLITS['Test'][1])

table[corrected] = np.nan
coef_rows = []
for h, g in table[is_val].groupby(level='horizon'):
    g = g.dropna(subset=[raw_bench, 'y_true'])
    b, a = np.polyfit(g[raw_bench], g['y_true'], 1)
    coef_rows.append({'Horizon': f'{h}h', 'a': a, 'b': b})
    rows = table.index.get_level_values('horizon') == h
    table.loc[rows, corrected] = np.clip(a + b * table.loc[rows, raw_bench], 0, None)
display(pd.DataFrame(coef_rows).round(3))

## III. Tập mẫu so sánh chung trên Test

Chỉ giữ các lần phát hành mà **mọi** phương pháp đều có dự báo. Khi có CAMS từ ADS,
đó là các lần 07:00 và 19:00 (giờ khởi tạo 00 UTC và 12 UTC của CAMS).

In [ ]:
METHODS = list(MODELS) + [c for c in ['CAMS forecast', 'Open-Meteo (CAMS)', corrected, 'Persistence'] if c in table]
test = table[is_test].dropna(subset=['y_true'] + METHODS)
if HAS_CAMS:
    test = test[test.index.get_level_values('issue_time').hour.isin([7, 19])]
# Giữ lần phát hành có đủ cả 4 horizon
counts = test.groupby(level='issue_time').size()
test = test[test.index.get_level_values('issue_time').isin(counts[counts == len(fc.HORIZONS)].index)]
print('Số lần phát hành:', test.index.get_level_values('issue_time').nunique(), '| số dòng:', len(test))

## IV. Kết quả: mô hình tốt hơn bao nhiêu %?

In [ ]:
def scores(g):
    out = {}
    for m in METHODS:
        e = g['y_true'] - g[m]
        out[(m, 'MAE')] = e.abs().mean()
        out[(m, 'RMSE')] = np.sqrt((e ** 2).mean())
    return pd.Series(out)

by_h = test.groupby(level='horizon').apply(scores)
by_h.index = [f'{h}h' for h in by_h.index]
by_h.loc['All'] = scores(test)
mae = by_h.xs('MAE', axis=1, level=1)[METHODS]
rmse = by_h.xs('RMSE', axis=1, level=1)[METHODS]
mae.to_csv(REPORT_DIR / 'benchmark_mae.csv'); rmse.to_csv(REPORT_DIR / 'benchmark_rmse.csv')
print('MAE (µg/m³)'); display(mae.round(2))
print('RMSE (µg/m³)'); display(rmse.round(2))

In [ ]:
# % giảm sai số = (MAE mốc - MAE mô hình) / MAE mốc. Dương = mô hình của ta tốt hơn.
benchmarks = [m for m in METHODS if m not in MODELS]
improve = pd.concat({model: ((mae[benchmarks].T - mae[model]) / mae[benchmarks].T * 100).T
                     for model in MODELS}, axis=1)
improve.to_csv(REPORT_DIR / 'benchmark_improvement_pct.csv')
display(improve.round(1).astype(str) + '%')

## V. Khác biệt có ý nghĩa thống kê? (Diebold–Mariano)

In [ ]:
dm_rows = []
for model in MODELS:
    for bench in benchmarks:
        for h, g in test.groupby(level='horizon'):
            g = g.sort_index()
            stat, p = fc.diebold_mariano(g['y_true'] - g[model], g['y_true'] - g[bench], int(h))
            dm_rows.append({'Model': model, 'Mốc': bench, 'Horizon': f'{h}h', 'DM_stat': stat, 'p_value': p,
                            'Kết luận': (f'{model} tốt hơn' if stat < 0 else f'{bench} tốt hơn') if p < 0.05
                                        else 'Không khác biệt rõ'})
dm = pd.DataFrame(dm_rows)
dm.to_csv(REPORT_DIR / 'benchmark_diebold_mariano.csv', index=False)
display(dm.round(4))

## VI. Biểu đồ cho báo cáo

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
hs = [f'{h}h' for h in fc.HORIZONS]
for m in METHODS:
    ax.plot(hs, mae.loc[hs, m], marker='o', lw=2.5 if m in MODELS else 1.2,
            ls='-' if m in MODELS else '--', label=m)
ax.set_xlabel('Horizon'); ax.set_ylabel('MAE (µg/m³)')
ax.set_title('Sai số dự báo PM2.5 Hoàn Kiếm trên Test')
ax.legend(); plt.tight_layout()
plt.savefig(REPORT_DIR / 'benchmark_mae_by_horizon.png', dpi=200); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(12, 4), sharey=True)
for ax, model in zip(axes, MODELS):
    data_pct = improve[model].loc[hs]
    data_pct.plot.bar(ax=ax, rot=0)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{model}: % giảm MAE so với từng mốc'); ax.set_ylabel('%')
plt.tight_layout(); plt.savefig(REPORT_DIR / 'benchmark_improvement.png', dpi=200); plt.show()

In [ ]:
# Một tháng mùa đông, dự báo trước 24 giờ
h = 24
g = test.xs(h, level='horizon').set_index('target_time').loc['2025-12-01':'2025-12-31']
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(g.index, g['y_true'], color='black', lw=1.5, label='Thực tế')
for m in METHODS:
    if m != 'Persistence':
        ax.plot(g.index, g[m], lw=1, marker='.' if HAS_CAMS else None, label=m)
ax.set_ylabel('PM2.5 (µg/m³)'); ax.set_title(f'Dự báo trước {h} giờ, tháng 12/2025'); ax.legend()
plt.tight_layout(); plt.savefig(REPORT_DIR / 'benchmark_timeseries_24h.png', dpi=200); plt.show()

In [ ]:
best = {m: mae.loc['All', m] for m in METHODS}
for model in MODELS:
    for bench in benchmarks:
        pct = (best[bench] - best[model]) / best[bench] * 100
        word = 'thấp hơn' if pct > 0 else 'cao hơn'
        print(f'{model}: MAE {best[model]:.2f} µg/m³, {word} {abs(pct):.1f}% so với {bench} ({best[bench]:.2f})')